In [0]:
storage_key = dbutils.secrets.get(scope="kv-finbank", key="storage-account-key")
spark.conf.set("fs.azure.account.key.stfinbankdevfbcq2026.dfs.core.windows.net",storage_key)

In [0]:
SILVER_BASE = "abfss://silver@stfinbankdevfbcq2026.dfs.core.windows.net"
GOLD_BASE = "abfss://gold@stfinbankdevfbcq2026.dfs.core.windows.net"
 
try:
    filtro_tabla = dbutils.widgets.get("filtro_tabla")
except Exception:
    filtro_tabla = ""
 
def _debe_procesar(*nombres_tabla_origen):
    return filtro_tabla == "" or filtro_tabla in nombres_tabla_origen
 
print(f"se ejecuta unicamente: '{filtro_tabla}'"
    if filtro_tabla
    else "se procesaran todas las tablas"
)

In [0]:
def construir_dim_clientes():
    from pyspark.sql.functions import col
 
    df_silver = spark.read.format("delta").load(f"{SILVER_BASE}/clientes")

    df_gold = df_silver.select(
        col("id_cli"),
        col("nombre_completo_hash"),
        col("tip_doc"),
        col("num_doc_hash"),
        col("fec_nac"),
        col("edad"),
        col("fec_alta"),
        col("cod_segmento").alias("segmento_legible"),
        col("score_buro"),
        col("score_buro_nulo"),
        col("ciudad_res"),
        col("depto_res"),
        col("estado_cli"),
        col("canal_adquis"),
    )
 
    gold_path = f"{GOLD_BASE}/dim_clientes"
    df_gold.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
        .partitionBy("segmento_legible").save(gold_path)
    print(f"gold.dim_clientes escrito: {df_gold.count()} filas, particionado por segmento_legible")
    return df_gold
 
 
if _debe_procesar("TB_CLIENTES_CORE"):
    df_dim_clientes = construir_dim_clientes()
    display(df_dim_clientes.limit(5))
else:
    print("se omite esta tabla")

In [0]:
def construir_dim_productos():
    from pyspark.sql.functions import col

    df_silver = spark.read.format("delta").load(f"{SILVER_BASE}/productos")
    df_gold = (
        df_silver
        .withColumnRenamed("desc_prod", "descripcion_producto")
        .withColumnRenamed("tip_prod", "tipo_producto")
        .select(
            "cod_prod", "descripcion_producto", "tipo_producto", "familia_producto",
            "tasa_ea", "tasa_mensual_equiv", "plazo_max_meses",
            "cuota_min", "comision_admin", "estado_prod",
        )
    )
 
    gold_path = f"{GOLD_BASE}/dim_productos"
    df_gold.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(gold_path)
    print(f"gold.dim_productos escrito: {df_gold.count()} filas")
    return df_gold
 
 
if _debe_procesar("TB_PRODUCTOS_CAT"):
    df_dim_productos = construir_dim_productos()
    display(df_dim_productos.limit(5))
else:
    print("se omite esta tabla")

In [0]:
def construir_dim_geografia():
    from pyspark.sql.functions import monotonically_increasing_id, col

    df_silver = spark.read.format("delta").load(f"{SILVER_BASE}/sucursales")
 
    df_gold = (
        df_silver
        .select("ciudad", "depto")
        .distinct()
        .withColumn("id_geografia", monotonically_increasing_id())
        .select("id_geografia", "ciudad", "depto")
    )
 
    gold_path = f"{GOLD_BASE}/dim_geografia"
    df_gold.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(gold_path)
    print(f"gold.dim_geografia escrito: {df_gold.count()} filas")
    return df_gold
 
 
def construir_dim_canal():
    from pyspark.sql.functions import monotonically_increasing_id, col, when
    df_silver = spark.read.format("delta").load(f"{SILVER_BASE}/sucursales")
 
    df_gold = (
        df_silver
        .select("tip_punto")
        .distinct()
        .withColumn(
            "es_canal_digital",
            when(col("tip_punto") == "Punto Digital", True).otherwise(False)
        )
        .withColumn("id_canal", monotonically_increasing_id())
        .select("id_canal", "tip_punto", "es_canal_digital")
    )
    gold_path = f"{GOLD_BASE}/dim_canal"
    df_gold.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(gold_path)
    print(f"gold.dim_canal escrito: {df_gold.count()} filas")
    return df_gold
 
if _debe_procesar("TB_SUCURSALES_RED"):
    df_dim_geografia = construir_dim_geografia()
    display(df_dim_geografia)
 
    df_dim_canal = construir_dim_canal()
    display(df_dim_canal)
else:
    print("Se omite dim_geografia dim_canal")

In [0]:
def construir_fact_transacciones():
    from pyspark.sql.functions import col, when, hour, date_format

    df_silver = spark.read.format("delta").load(f"{SILVER_BASE}/movimientos_financieros")

    TASA_COP_USD = 3200
 
    df_gold = (
        df_silver
        .withColumn("vr_mov_usd", (col("vr_mov") / TASA_COP_USD))
        .withColumn(
            "flag_horario_habil",
            when((hour(col("hra_mov")) >= 8) & (hour(col("hra_mov")) <= 18), True).otherwise(False)
        )
        .withColumn("periodo_particion", date_format(col("fec_mov"), "yyyy-MM"))
        .select(
            "id_mov", "id_cli", "cod_prod", "fec_mov", "hra_mov",
            "vr_mov", "vr_mov_usd", "tip_mov", "cod_canal", "cod_ciudad",
            "cod_estado_mov",
            "promedio_movil_30d", "stddev_movil_30d", "ind_sospechoso",
            "flag_horario_habil", "periodo_particion",
        )
    )
 
    gold_path = f"{GOLD_BASE}/fact_transacciones"
    df_gold.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
        .partitionBy("periodo_particion").save(gold_path)
    print(f"gold.fact_transacciones escrito: {df_gold.count()} filas (particionado por periodo_particion)")
    return df_gold
 
 
if _debe_procesar("TB_MOV_FINANCIEROS"):
    df_fact_transacciones = construir_fact_transacciones()
    display(df_fact_transacciones.limit(5))
else:
    print("se omite esta tabla")

In [0]:
def construir_fact_cartera():
    from pyspark.sql.functions import col, when, lit
    df_silver = spark.read.format("delta").load(f"{SILVER_BASE}/obligaciones")
 
    df_con_bucket = df_silver.withColumn(
        "bucket_mora",
        when(col("dias_mora_act") == 0, "Al dia")
        .when((col("dias_mora_act") >= 1) & (col("dias_mora_act") <= 30), "Rango 1")
        .when((col("dias_mora_act") >= 31) & (col("dias_mora_act") <= 60), "Rango 2")
        .when((col("dias_mora_act") >= 61) & (col("dias_mora_act") <= 90), "Rango 3")
        .otherwise("Deteriorado")
    )
    df_con_clasificacion = df_con_bucket.withColumn(
        "clasificacion_regulatoria",
        when(col("bucket_mora") == "Al dia", "A")
        .when(col("bucket_mora") == "Rango 1", "B")
        .when(col("bucket_mora") == "Rango 2", "C")
        .when(col("bucket_mora") == "Rango 3", "D")
        .otherwise("E")
    ).withColumn(
        "pct_provision",
        when(col("bucket_mora") == "Al dia", 0.01)
        .when(col("bucket_mora") == "Rango 1", 0.05)
        .when(col("bucket_mora") == "Rango 2", 0.20)
        .when(col("bucket_mora") == "Rango 3", 0.50)
        .otherwise(1.00)
    ).withColumn(
        "provision_estimada", col("sdo_capital") * col("pct_provision")
    )
 
    df_gold = df_con_clasificacion.select(
        "id_oblig", "id_cli", "cod_prod", "vr_aprobado", "vr_desembolsado",
        "sdo_capital", "vr_cuota", "fec_desembolso", "fec_venc",
        "dias_mora_act", "bucket_mora", "clasificacion_regulatoria",
        "provision_estimada", "num_cuotas_pend", "calif_riesgo",
    )
 
    gold_path = f"{GOLD_BASE}/fact_cartera"
    df_gold.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
        .partitionBy("bucket_mora").save(gold_path)
    print(f"gold.fact_cartera escrito: {df_gold.count()} filas")
 
    print("Distribucion de bucket_mora:")
    df_gold.groupBy("bucket_mora").count().orderBy("bucket_mora").show()
 
    return df_gold
if _debe_procesar("TB_OBLIGACIONES"):
    df_fact_cartera = construir_fact_cartera()
    display(df_fact_cartera.limit(5))
else:
    print("se omite esta tabla")

In [0]:
def construir_fact_rentabilidad_cliente():
    from pyspark.sql.functions import col, sum as spark_sum, date_format, lit

    df_movimientos = spark.read.format("delta").load(f"{SILVER_BASE}/movimientos_financieros")
    df_comisiones = spark.read.format("delta").load(f"{SILVER_BASE}/comisiones")
    df_intereses = (
        df_movimientos
        .filter(col("tip_mov") == "pago_interes")
        .withColumn("periodo_mes", date_format(col("fec_mov"), "yyyy-MM"))
        .groupBy("id_cli", "periodo_mes")
        .agg(spark_sum("vr_mov").alias("ingreso_intereses"))
    )
    df_comisiones_cobradas = (
        df_comisiones
        .filter(col("estado_cobro") == "Cobrado")
        .withColumn("periodo_mes", date_format(col("fec_cobro"), "yyyy-MM"))
        .groupBy("id_cli", "periodo_mes")
        .agg(spark_sum("vr_comision").alias("ingreso_comisiones"))
    )
 
    df_join = df_intereses.join(
        df_comisiones_cobradas, on=["id_cli", "periodo_mes"], how="outer"
    ).fillna(0, subset=["ingreso_intereses", "ingreso_comisiones"])
 
    df_gold = df_join.withColumn(
        "ingreso_total", col("ingreso_intereses") + col("ingreso_comisiones")
    )
 
    gold_path = f"{GOLD_BASE}/fact_rentabilidad_cliente"
    df_gold.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
        .partitionBy("periodo_mes").save(gold_path)
    print(f"gold.fact_rentabilidad_cliente escrito: {df_gold.count()} filas")
 
    return df_gold
 
 
if _debe_procesar("TB_MOV_FINANCIEROS", "TB_COMISIONES_LOG"):
    df_fact_rentabilidad = construir_fact_rentabilidad_cliente()
    display(df_fact_rentabilidad.limit(10))
else:
    print("se omite esta tabla")

In [0]:
def construir_agg_mora_por_segmento_region():
    from pyspark.sql.functions import col, sum as spark_sum, count, when, round as spark_round

    df_cartera = spark.read.format("delta").load(f"{GOLD_BASE}/fact_cartera")
    df_clientes = spark.read.format("delta").load(f"{GOLD_BASE}/dim_clientes")
 
    df_join = df_cartera.join(
        df_clientes.select("id_cli", "segmento_legible", "ciudad_res"), on="id_cli", how="left"
    )
 
    df_agg = (
        df_join
        .groupBy("segmento_legible", "ciudad_res")
        .agg(
            count("id_oblig").alias("total_obligaciones"),
            spark_sum("sdo_capital").alias("monto_total_cartera"),
            spark_sum(when(col("bucket_mora") != "Al dia", col("sdo_capital")).otherwise(0)).alias("monto_en_mora"),
        )
        .withColumn(
            "tasa_mora_pct",
            spark_round(col("monto_en_mora") / col("monto_total_cartera") * 100, 2)
        )
    )
 
    gold_path = f"{GOLD_BASE}/agg_mora_por_segmento_region"
    df_agg.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(gold_path)
    print(f"gold.agg_mora_por_segmento_region escrito: {df_agg.count()} filas")
    return df_agg
 
 
if filtro_tabla == "":
    df_agg_mora = construir_agg_mora_por_segmento_region()
    display(df_agg_mora)
else:
    print("solo se recalcula en corrida completa")

In [0]:
def construir_agg_transacciones_por_canal_ciudad():
    from pyspark.sql.functions import col, count, sum as spark_sum, avg, round as spark_round

    df_trans = spark.read.format("delta").load(f"{GOLD_BASE}/fact_transacciones")
 
    df_agg = (
        df_trans
        .groupBy("cod_canal", "cod_ciudad")
        .agg(
            count("id_mov").alias("total_transacciones"),
            spark_sum("vr_mov").alias("monto_total"),
            spark_round(avg("vr_mov"), 2).alias("monto_promedio"),
            spark_sum(col("ind_sospechoso")).alias("total_sospechosas"),
        )
    )
 
    gold_path = f"{GOLD_BASE}/agg_transacciones_por_canal_ciudad"
    df_agg.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(gold_path)
    print(f"gold.agg_transacciones_por_canal_ciudad escrito: {df_agg.count()} filas")
    return df_agg
 
 
if filtro_tabla == "":
    df_agg_transacciones = construir_agg_transacciones_por_canal_ciudad()
    display(df_agg_transacciones)
else:
    print("solo se recalcula en corrida completa")

In [0]:
def construir_cltv_12m():
    from pyspark.sql.functions import sum as spark_sum

    df_rentabilidad = spark.read.format("delta").load(f"{GOLD_BASE}/fact_rentabilidad_cliente")
    df_cltv = (
        df_rentabilidad
        .groupBy("id_cli")
        .agg(
            spark_sum("ingreso_intereses").alias("total_ingreso_intereses_12m"),
            spark_sum("ingreso_comisiones").alias("total_ingreso_comisiones_12m"),
            spark_sum("ingreso_total").alias("cltv_12m"),
        )
    )
 
    gold_path = f"{GOLD_BASE}/cltv_12m"
    df_cltv.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(gold_path)
    print(f"gold.cltv_12m escrito: {df_cltv.count()} filas,1 por cliente")
    return df_cltv 
if filtro_tabla == "":
    df_cltv = construir_cltv_12m()
    display(df_cltv.orderBy(df_cltv["cltv_12m"].desc()).limit(10))
else:
    print("solo se recalcula en corrida completa")

In [0]:
def construir_agg_vista_comercial_cliente():
    from pyspark.sql.functions import col, count, min as spark_min

    df_clientes = spark.read.format("delta").load(f"{GOLD_BASE}/dim_clientes")
    df_obligaciones = spark.read.format("delta").load(f"{GOLD_BASE}/fact_cartera")
    df_transacciones = spark.read.format("delta").load(f"{GOLD_BASE}/fact_transacciones")
    df_cltv = spark.read.format("delta").load(f"{GOLD_BASE}/cltv_12m")
 
    df_uso_obligaciones = df_obligaciones.groupBy("id_cli").agg(count("id_oblig").alias("num_obligaciones"))
    df_uso_transacciones = df_transacciones.groupBy("id_cli").agg(count("id_mov").alias("num_transacciones"))
    
    from pyspark.sql.functions import expr
    orden_riesgo = expr(
        "CASE bucket_mora "
        "WHEN 'Deteriorado' THEN 5 WHEN 'Rango 3' THEN 4 WHEN 'Rango 2' THEN 3 "
        "WHEN 'Rango 1' THEN 2 ELSE 1 END"
    )
    df_peor_bucket = (
        df_obligaciones
        .withColumn("orden_riesgo", orden_riesgo)
        .groupBy("id_cli")
        .agg({"orden_riesgo": "max"})
        .withColumnRenamed("max(orden_riesgo)", "orden_riesgo_max")
    )
 
    df_gold = (
        df_clientes.select("id_cli", "nombre_completo_hash", "segmento_legible", "ciudad_res", "estado_cli")
        .join(df_uso_obligaciones, on="id_cli", how="left")
        .join(df_uso_transacciones, on="id_cli", how="left")
        .join(df_cltv.select("id_cli", "cltv_12m"), on="id_cli", how="left")
        .join(df_peor_bucket, on="id_cli", how="left")
        .fillna(0, subset=["num_obligaciones", "num_transacciones", "cltv_12m"])
    )
 
    gold_path = f"{GOLD_BASE}/agg_vista_comercial_cliente"
    df_gold.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(gold_path)
    print(f"gold.agg_vista_comercial_cliente escrito: {df_gold.count()} filas")
 
    spark.sql(f"OPTIMIZE delta.`{gold_path}` ZORDER BY (segmento_legible, ciudad_res)")
    return df_gold
 
if filtro_tabla == "":
    df_vista_comercial = construir_agg_vista_comercial_cliente()
    display(df_vista_comercial.limit(10))
else:
    print("solo se recalcula en corrida completa")

In [0]:
def construir_kpis_cartera_diarios():
    from pyspark.sql.functions import col, count, sum as spark_sum, when, round as spark_round, current_date, countDistinct

    df_cartera = spark.read.format("delta").load(f"{GOLD_BASE}/fact_cartera")
    df_clientes = spark.read.format("delta").load(f"{GOLD_BASE}/dim_clientes")
 
    df_join = df_cartera.join(
        df_clientes.select("id_cli", "segmento_legible", "ciudad_res"), on="id_cli", how="left"
    )
 
    df_kpis = (
        df_join
        .withColumn("fecha", current_date())
        .groupBy("fecha", "cod_prod", "segmento_legible", "ciudad_res")
        .agg(
            count("id_oblig").alias("total_obligaciones_activas"),
            spark_sum("sdo_capital").alias("monto_total_cartera"),
            spark_sum(when(col("bucket_mora") != "Al dia", col("sdo_capital")).otherwise(0)).alias("monto_en_mora"),
            countDistinct(when(col("bucket_mora") != "Al dia", col("id_cli"))).alias("clientes_en_mora"),
        )
        .withColumn(
            "tasa_mora_pct",
            spark_round(col("monto_en_mora") / col("monto_total_cartera") * 100, 2)
        )
    )
 
    gold_path = f"{GOLD_BASE}/kpis_cartera_diarios"
    df_kpis.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(gold_path)
    print(f"gold.kpis_cartera_diarios escrito: {df_kpis.count()} filas")
 
    spark.sql(f"OPTIMIZE delta.`{gold_path}` ZORDER BY (segmento_legible, ciudad_res)")
    print("  OPTIMIZE ZORDER aplicado")
 
    return df_kpis
 
 
if filtro_tabla == "":
    df_kpis = construir_kpis_cartera_diarios()
    display(df_kpis.limit(10))
else:
    print("solo se recalcula en corrida completa")